Outline:

- Download qiskit circuit cutting addon
    - run `pip install qiskit-addon-cutting` in your conda or venv environment
- Find a large circuit, show it can't run on our quantum computer
- See how different circuit sizes, depths, and target QPU sizes affect overhead and output depth
- Show how to do the actual cutting and run it on the quantum computer
    - oops this library doesn't actually support cutting circuits with classical measurements
        - all observations and expectation values
        - seemingly designed more towards physics/chemistry sorts of problems rather than arbitrary computation?
        - https://github.com/Qiskit/qiskit-addon-cutting/pull/428 PR for partial support that hasn't been touched in 2 years :<

In [369]:
# for convenience 
# ! pip install qiskit-addon-cutting

## Making and Cutting Large Circuits

In [370]:
import numpy as np
from qiskit.circuit.random import random_circuit
from qiskit.quantum_info import SparsePauliOp
from qiskit import QuantumCircuit

from qiskit.transpiler import TranspilerError, generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService

# set up your backend so it knows to try to fit it into the quantum computer we have here, your service may be named differently
service = QiskitRuntimeService(name="DQC")
backend = service.backend("ibm_rensselaer")

In [371]:
from typing import Hashable

from qiskit_addon_cutting.automated_cut_finding import (
    find_cuts,
    OptimizationParameters,
    DeviceConstraints,
)
from qiskit_addon_cutting import cut_wires, expand_observables, partition_problem
from qiskit.quantum_info import PauliList

type CutCircuitInfo = tuple[QuantumCircuit, int, float, QuantumCircuit, dict[str, float], SparsePauliOp, dict[Hashable, QuantumCircuit], dict[Hashable, PauliList]]

def makeAndCutRandomCircuit(num_qubits: int, depth: int, qubits_in_qpu: int, seed=144) -> CutCircuitInfo:
    """
    Generates a random circuit with the given parameters and attempts to cut it into subcircuits to fit the given QPU size

    :return: tuple[original circuit, number of cuts, sampling overhead, cut circuit, full cut metadata (for printing), subcircuits, subobservables (for later)] 
    """
    qc = random_circuit(num_qubits, depth, max_operands=2, seed=seed)
    
    # as far as i can tell this corresponds vaguely to "measure each qubit in the standard 0/1 basis"
    observable = SparsePauliOp(["Z"*num_qubits])
    # these might be more correct? not sure
    # observable = SparsePauliOp([ ''.join([("Z" if i == o else "I") for i in range(num_qubits)]) for o in range(num_qubits)])
    # observable = SparsePauliOp([(format(o, f'0{num_qubits}b')).replace("0", "I").replace("1", "Z") for o in range(2**num_qubits)])

    # find an optimal cut for our circuit (minimize sampling overhead i think?)
    optimization_settings = OptimizationParameters(seed)
    device_constraints = DeviceConstraints(qubits_per_subcircuit=qubits_in_qpu)
    cut_circuit, metadata = find_cuts(qc, optimization_settings, device_constraints)

    # each cut qubit wire needs to exist in both subcircuits it'll show up in, so add a new qubit for that
    qc_w_ancilla = cut_wires(cut_circuit)
    observables_expanded = expand_observables(observable.paulis, qc, qc_w_ancilla)
    
    # actually separate the cut circuit into subcircuits with their own corresponding observables
    partitioned_problem = partition_problem(
        circuit=qc_w_ancilla, observables=observables_expanded
    )

    subcircuits = partitioned_problem.subcircuits
    subobservables = partitioned_problem.subobservables
    
    return (qc, len(metadata["cuts"]), metadata["sampling_overhead"], cut_circuit, metadata, observable, subcircuits, subobservables)

In [372]:
# try generating some circuits! see what their overheads end up as. mess around with the parameters

circuit_cut_res = makeAndCutRandomCircuit(num_qubits=30, depth=4, qubits_in_qpu=3)

qc, cut_count, sampling_overhead, _, metadata, observable, subcircuits, subobservables = circuit_cut_res

# check if it can fit on our quantum computer (should be equivalent to just checking if the qubit count is 127 or below)
pass_manager = generate_preset_pass_manager(optimization_level=1, backend=backend)
try:
    isa_qc = pass_manager.run(qc)
except TranspilerError as e:
    print(e)

print(
    f'Circuit has {len(qc.qubits)} qubits, {sum(qc.count_ops().values())} operations, and depth {qc.depth()}\n'
    f'Starting out with {qc.num_connected_components()} connected components\n'
    f'Using {cut_count} cut(s) to divide the circuit into {len(subcircuits)} subcircuits\n'
    f'This has a sampling overhead of {metadata["sampling_overhead"]}\n'
    f'Subcircuit info:'
)

for scl in subcircuits:
    sc = subcircuits[scl]
    print(f'\t{scl}: subcircuit with {len(sc.qubits)} qubits, {sum(sc.count_ops().values())} operations, and depth {sc.depth()}')


Circuit has 30 qubits, 91 operations, and depth 4
Starting out with 5 connected components
Using 13 cut(s) to divide the circuit into 15 subcircuits
This has a sampling overhead of 8232690495.875816
Subcircuit info:
	0: subcircuit with 3 qubits, 9 operations, and depth 3
	1: subcircuit with 3 qubits, 9 operations, and depth 4
	2: subcircuit with 2 qubits, 7 operations, and depth 4
	3: subcircuit with 2 qubits, 7 operations, and depth 4
	4: subcircuit with 2 qubits, 7 operations, and depth 2
	5: subcircuit with 3 qubits, 10 operations, and depth 4
	6: subcircuit with 3 qubits, 10 operations, and depth 3
	7: subcircuit with 1 qubits, 4 operations, and depth 3
	8: subcircuit with 1 qubits, 3 operations, and depth 2
	9: subcircuit with 3 qubits, 10 operations, and depth 4
	10: subcircuit with 3 qubits, 10 operations, and depth 4
	11: subcircuit with 2 qubits, 7 operations, and depth 4
	12: subcircuit with 1 qubits, 4 operations, and depth 4
	13: subcircuit with 1 qubits, 4 operations, and 

## Run it on the quantum computer

In [373]:
from qiskit_addon_cutting import generate_cutting_experiments

# do all the measurements / input varying 
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits, observables=subobservables, num_samples=1_000
)

# compile down as usual
isa_subexperiments = {
    label: pass_manager.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [ ]:
from qiskit_ibm_runtime import SamplerV2, Batch

# send everything in a batch to the quantum computer (uncomment when you're ready to send stuff, try not to spam it too hard, some of these can get expensive)

# with Batch(backend=backend) as batch:
#     sampler = SamplerV2(mode=batch)
#     jobs = {
#         label: sampler.run(subsystem_subexpts, shots=2**12)
#         for label, subsystem_subexpts in isa_subexperiments.items()
#     }

In [ ]:
from qiskit_addon_cutting import reconstruct_expectation_values

results = {label: job.result() for label, job in jobs.items()}

reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, observable.coeffs)


In [ ]:
for i in range(len(reconstructed_expval_terms)):
    print(f'Gate: {observable[i]}: {reconstructed_expval_terms[i]}')
    

Gate: SparsePauliOp(['ZZZZ'],
              coeffs=[1.+0.j]): 0.8815560340881348
